# Summly RAG Evaluation — RAGAS Metrics

Evaluates the Summly RAG system using three metrics:
- **Faithfulness** — do answers contradict the meeting transcript?
- **Answer Relevancy** — are answers actually about what was asked?
- **Context Recall** — does hybrid search retrieve the right chunks?

Golden dataset: 10 Q/A/context triples from meeting transcripts.
Judge LLM: Groq llama-3.3-70b | Embeddings: sentence-transformers/all-MiniLM-L6-v2

In [48]:
# Quick diagnostic — shows every metric available in your exact install
import ragas.metrics.collections as col
print([x for x in dir(col) if not x.startswith("_")])

['AgentGoalAccuracy', 'AgentGoalAccuracyWithReference', 'AgentGoalAccuracyWithoutReference', 'AnswerAccuracy', 'AnswerCorrectness', 'AnswerRelevancy', 'BaseMetric', 'BleuScore', 'CHRFScore', 'ContextEntityRecall', 'ContextPrecision', 'ContextPrecisionWithReference', 'ContextPrecisionWithoutReference', 'ContextRecall', 'ContextRelevance', 'ContextUtilization', 'DataCompyScore', 'DistanceMeasure', 'DomainSpecificRubrics', 'ExactMatch', 'FactualCorrectness', 'Faithfulness', 'InstanceSpecificRubrics', 'MultiModalFaithfulness', 'MultiModalRelevance', 'NoiseSensitivity', 'NonLLMStringSimilarity', 'QuotedSpansAlignment', 'ResponseGroundedness', 'RougeScore', 'RubricsScoreWithReference', 'RubricsScoreWithoutReference', 'SQLSemanticEquivalence', 'SemanticSimilarity', 'StringPresence', 'SummaryScore', 'ToolCallAccuracy', 'ToolCallF1', 'TopicAdherence', 'agent_goal_accuracy', 'answer_accuracy', 'answer_correctness', 'answer_relevancy', 'base', 'chrf_score', 'context_entity_recall', 'context_preci

In [49]:
# Cell 2 — Imports (correct for RAGAS 0.4.3)
import json, sys, os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# ── Path setup ───────────────────────────────────────────────────────
notebook_dir = os.path.abspath(".")
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Load .env ────────────────────────────────────────────────────────
load_dotenv(dotenv_path=os.path.join(project_root, ".env"))
groq_key = os.getenv("GROQ_API_KEY")
print(f"{'✓' if groq_key else '✗'} GROQ_API_KEY: {groq_key[:8] + '...' if groq_key else 'NOT FOUND'}")

# ── RAGAS 0.4.3 correct imports ──────────────────────────────────────
# 0.4.x uses EvaluationDataset, not HuggingFace Dataset
# Metric names changed: context_recall → LLMContextRecall, etc.
from ragas import evaluate, EvaluationDataset
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasHFEmbeddings

# These are the correct 0.4.3 metric class names
# Replace the import line with these exact names:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRecall


print("✓ All imports successful (RAGAS 0.4.3 API)")

✓ GROQ_API_KEY: gsk_Gvw2...
✓ All imports successful (RAGAS 0.4.3 API)


In [76]:
# Cell 3 — AsyncOpenAI client (required for agenerate to work)
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasHFEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRecall

groq_key = os.getenv("GROQ_API_KEY")

# AsyncOpenAI — not OpenAI — this is the fix
groq_client = AsyncOpenAI(
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
)

judge_llm = llm_factory(
    model="llama-3.1-8b-instant",
    provider="openai",
    client=groq_client,
)
print("✓ Judge LLM ready (async)")

print("Loading embeddings...")
embeddings_model = RagasHFEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
print("✓ Embeddings ready")

faithfulness_metric     = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(
    llm=judge_llm,
    embeddings=embeddings_model,
    strictness=1,
)
context_recall_metric   = ContextRecall(llm=judge_llm)

print("✓ All 3 metrics ready")

✓ Judge LLM ready (async)
Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Embeddings ready
✓ All 3 metrics ready


In [51]:
# Cell 4 — Load golden dataset
golden_path = os.path.join(notebook_dir, "golden_dataset.json")

with open(golden_path, "r") as f:
    golden_data = json.load(f)

print(f"✓ Loaded {len(golden_data)} question/answer/context triples")
print(f"\nFirst item preview:")
print(f"  Question : {golden_data[0]['question']}")
print(f"  Answer   : {golden_data[0]['ground_truth']}")
print(f"  Context  : {golden_data[0]['contexts'][0][:80]}...")

✓ Loaded 10 question/answer/context triples

First item preview:
  Question : What is the main concern discussed in the meeting?
  Answer   : The main concern is students skipping school, particularly on Fridays.
  Context  : I have our list of chronically absent students here and I've been noticing a tro...


In [52]:
# Cell 5 — Build evaluation records
#
# RAGAS needs exactly 4 fields per record:
#   question      → the question asked
#   answer        → what the RAG system said
#   contexts      → list of text chunks retrieved (from hybrid_search)
#   ground_truth  → the correct answer (from your golden dataset)

eval_records = []

# ── OPTION A: Real Summly answers (uncomment when you have meetings in DB) ──
# from server.core.rag.chat import chat_with_meeting
# MEETING_ID = 1   # change to your actual meeting ID
#
# for item in golden_data:
#     result = chat_with_meeting(query=item["question"], meeting_id=MEETING_ID)
#     eval_records.append({
#         "question":     item["question"],
#         "answer":       result["answer"],
#         "contexts":     [chunk["text"] for chunk in result["sources"]],
#         "ground_truth": item["ground_truth"],
#     })
#     print(f"  ✓ {item['question'][:55]}")

# ── OPTION B: Fake answers — use this to verify pipeline works ──────
# One answer is deliberately wrong (Alice instead of Bob) so we can
# confirm faithfulness correctly drops for bad answers.
fake_answers = {
    "When was the deployment date moved to?":
        "The deployment was moved to Q2.",
    "Who is responsible for sending the project proposal to the client?":
        "Alice will send the proposal.",    # ← deliberately wrong (Bob should be)
    "What is Sarah's task?":
        "Sarah is reviewing the Q3 budget and presenting findings next Monday.",
    "What should be prioritised — mobile app or desktop version?":
        "The mobile app should be prioritised.",
    "What will James do by end of week?":
        "James will schedule a follow-up meeting with the design team.",
    "Why was the deployment pushed back?":
        "More testing time was needed.",
    "What hiring decision was made?":
        "Two backend engineers will be hired.",
    "What did Alice present?":
        "Alice presented new feature proposals.",
    "What is Bob's deadline?":
        "Bob's deadline is this Friday.",
    "What was the main topic of the meeting?":
        "The Q3 product roadmap was discussed.",
}

for item in golden_data:
    answer = fake_answers.get(item["question"], "I could not find that in this meeting.")
    eval_records.append({
        "question":     item["question"],
        "answer":       answer,
        "contexts":     item["contexts"],
        "ground_truth": item["ground_truth"],
    })

print(f"✓ Built {len(eval_records)} evaluation records\n")

# Show preview table
preview_df = pd.DataFrame([{
    "Question": r["question"][:48] + "...",
    "Answer":   r["answer"][:48] + "...",
} for r in eval_records])
display(preview_df)

✓ Built 10 evaluation records



,Question,Answer
0,What is the main concern discussed in the meet...,I could not find that in this meeting....
1,Why are students finding it hard to come to sc...,I could not find that in this meeting....
2,What idea was suggested to encourage students ...,I could not find that in this meeting....
3,Why have students been getting sick?...,I could not find that in this meeting....
4,What symptoms did sick students have?...,I could not find that in this meeting....
5,What action was decided to help students avoid...,I could not find that in this meeting....
6,How many days has John Smith missed?...,I could not find that in this meeting....
7,Why has John Smith been missing school?...,I could not find that in this meeting....
8,What was suggested to help John Smith?...,I could not find that in this meeting....
9,What help was offered for John's family?...,I could not find that in this meeting....


In [53]:
# Cell 6 — Convert to RAGAS 0.4.3 EvaluationDataset
# 0.4.x uses EvaluationDataset with SingleTurnSample, not HuggingFace Dataset

from ragas.dataset_schema import SingleTurnSample

samples = []
for r in eval_records:
    samples.append(SingleTurnSample(
        user_input=r["question"],       # was "question"
        response=r["answer"],           # was "answer"
        retrieved_contexts=r["contexts"],  # was "contexts"
        reference=r["ground_truth"],    # was "ground_truth"
    ))

dataset = EvaluationDataset(samples=samples)

print(f"✓ EvaluationDataset ready: {len(dataset)} samples")
print(f"  Fields: user_input, response, retrieved_contexts, reference")

✓ EvaluationDataset ready: 10 samples
  Fields: user_input, response, retrieved_contexts, reference


In [77]:
# Cell 7 — Sync version, no async, no await — simplest possible
import time

scored = []
for i, sample in enumerate(dataset.samples):
    print(f"  Sample {i+1}/{len(dataset.samples)}...", end=" ", flush=True)
    
    try:
        faith  = faithfulness_metric.score(
            user_input=sample.user_input,
            response=sample.response,
            retrieved_contexts=sample.retrieved_contexts,
        )
        relev  = answer_relevancy_metric.score(
            user_input=sample.user_input,
            response=sample.response,
        )
        recall = context_recall_metric.score(
            user_input=sample.user_input,
            retrieved_contexts=sample.retrieved_contexts,
            reference=sample.reference,
        )

        f = float(faith.value)
        r = float(relev.value)
        c = float(recall.value)

        scored.append({
            "question":         sample.user_input,
            "faithfulness":     round(f, 3),
            "answer_relevancy": round(r, 3),
            "context_recall":   round(c, 3),
        })
        print(f"faith={f:.2f}  relev={r:.2f}  recall={c:.2f}")
    
    except Exception as e:
        print(f"SKIP — {str(e)[:60]}")
        scored.append({
            "question":         sample.user_input,
            "faithfulness":     None,
            "answer_relevancy": None,
            "context_recall":   None,
        })
    
    time.sleep(2)  # 2 second gap between samples

print(f"\n✓ Done! {len(scored)} samples scored.")

  Sample 1/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 2/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 3/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 4/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 5/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 6/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 7/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 8/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 9/10... SKIP — Cannot call sync score() from an async context. Use ascore()
  Sample 10/10... SKIP — Cannot call sync score() from an async context. Use ascore()

✓ Done! 10 samples scored.


In [70]:
# Cell 8 — Summary table
import pandas as pd

results_df = pd.DataFrame(scored)

faith_avg  = results_df["faithfulness"].mean()
relev_avg  = results_df["answer_relevancy"].mean()
recall_avg = results_df["context_recall"].mean()

def rating(s):
    if s >= 0.85: return "🟢 Good"
    if s >= 0.70: return "🟡 Fair"
    return "🔴 Needs work"

print("=" * 58)
print("      SUMMLY RAG EVALUATION — RAGAS SCORES")
print("=" * 58)
print(f"\n  Faithfulness:      {faith_avg:.3f}   {rating(faith_avg)}")
print(f"  Answer Relevancy:  {relev_avg:.3f}   {rating(relev_avg)}")
print(f"  Context Recall:    {recall_avg:.3f}   {rating(recall_avg)}")
print(f"\n  Overall Average:   {(faith_avg+relev_avg+recall_avg)/3:.3f}")
print("=" * 58)

print("\nPer-question breakdown:")
display(results_df[["question","faithfulness","answer_relevancy","context_recall"]])

      SUMMLY RAG EVALUATION — RAGAS SCORES

  Faithfulness:      0.100   🔴 Needs work
  Answer Relevancy:  0.000   🔴 Needs work
  Context Recall:    1.000   🟢 Good

  Overall Average:   0.367

Per-question breakdown:


,question,faithfulness,answer_relevancy,context_recall
0,What is the main concern discussed in the meet...,0.0,0.0,1.0
1,Why are students finding it hard to come to sc...,0.0,0.0,1.0
2,What idea was suggested to encourage students ...,0.0,0.0,1.0
3,Why have students been getting sick?,0.5,0.0,1.0
4,What symptoms did sick students have?,0.0,0.0,1.0
5,What action was decided to help students avoid...,0.0,0.0,1.0
6,How many days has John Smith missed?,0.0,0.0,1.0
7,Why has John Smith been missing school?,0.0,0.0,1.0
8,What was suggested to help John Smith?,0.5,0.0,1.0
9,What help was offered for John's family?,0.0,0.0,1.0


In [71]:
# Cell 9 — Save
import json

results_df.to_json("results.json", indent=2, orient="records")
print("✓ eval/results.json saved")

summary = {
    "faithfulness":     round(faith_avg,  4),
    "answer_relevancy": round(relev_avg,  4),
    "context_recall":   round(recall_avg, 4),
    "overall_avg":      round((faith_avg+relev_avg+recall_avg)/3, 4),
    "num_questions":    len(scored),
    "judge_model":      "groq/llama-3.3-70b-versatile",
    "embedding_model":  "sentence-transformers/all-MiniLM-L6-v2",
}

with open("summary_scores.json", "w") as f:
    json.dump(summary, f, indent=2)
print("✓ eval/summary_scores.json saved")

print(f"\n📌 Resume bullet scores:")
print(f"   Faithfulness {summary['faithfulness']} | "
      f"Relevancy {summary['answer_relevancy']} | "
      f"Recall {summary['context_recall']}")

✓ eval/results.json saved
✓ eval/summary_scores.json saved

📌 Resume bullet scores:
   Faithfulness 0.1 | Relevancy 0.0 | Recall 1.0


In [ ]:
# Cell 10 — Save results to files

# Detailed per-question results
results_path = os.path.join(notebook_dir, "results.json")
results_df.to_json(results_path, indent=2, orient="records")
print(f"✓ Detailed results → eval/results.json")

# Clean summary (for README / resume)
summary = {
    "faithfulness":     round(float(results["faithfulness"]), 4),
    "answer_relevancy": round(float(results["answer_relevancy"]), 4),
    "context_recall":   round(float(results["context_recall"]), 4),
    "num_questions":    len(eval_records),
    "judge_model":      "groq/llama-3.3-70b-versatile",
    "embedding_model":  "sentence-transformers/all-MiniLM-L6-v2",
}

summary_path = os.path.join(notebook_dir, "summary_scores.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Summary scores   → eval/summary_scores.json")

print(f"\n📌 Resume bullet scores:")
print(f"   Faithfulness {summary['faithfulness']} | Relevancy {summary['answer_relevancy']} | Context Recall {summary['context_recall']}")